
# Local Attention Window Ablation

This Colab notebook tests **Local Attention BERT** with window sizes:

- 3
- 5
- 7
- 9

on:
- SMS Spam Collection
- Enron spam/ham

## Output
This notebook saves and prints **F1 score only** for each window size and dataset.


## Cell 1 — Install packages

In [ ]:

!pip -q install transformers==4.39.3 scikit-learn pandas tqdm pyyaml


## Cell 2 — Mount Google Drive

In [ ]:

from google.colab import drive
drive.mount('/content/drive')


## Cell 3 — Imports

In [ ]:

import os
import re
import gc
import json
import math
import yaml
import random
import warnings
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import BertTokenizer, BertConfig, BertModel, BertPreTrainedModel, AdamW, get_linear_schedule_with_warmup
from transformers.models.bert.modeling_bert import BertSelfAttention

warnings.filterwarnings("ignore")


## Cell 4 — Config

In [ ]:

CONFIG = {
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    "sms_file": "/content/drive/MyDrive/sms/SMSSpamCollection",
    "enron_root": "/content/drive/MyDrive/enron1",
    "output_root": "/content/drive/MyDrive/local_attention_window_ablation",

    "train_size": 0.70,
    "val_size": 0.20,
    "test_size": 0.10,

    "batch_size": 16,
    "epochs": 6,
    "lr": 5e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "max_grad_norm": 1.0,

    "sms_max_len": 64,
    "enron_max_len": 128,

    "num_labels": 2,
    "window_sizes": [3, 5, 7, 9,],

    "vocab_size": 30522,
    "hidden_size": 768,
    "num_hidden_layers": 12,
    "num_attention_heads": 12,
    "intermediate_size": 3072,
    "hidden_dropout_prob": 0.1,
    "attention_probs_dropout_prob": 0.1,

    "allowed_extensions": [".txt", ".text", ".eml", ".msg"],
}

os.makedirs(CONFIG["output_root"], exist_ok=True)
print("Device:", CONFIG["device"])
print("Output root:", CONFIG["output_root"])


## Cell 5 — Reproducibility

In [ ]:

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])


## Cell 6 — Data helpers

In [ ]:

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        text = str(text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s@:/\.\-\$£€%!?]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def list_text_files(folder: str, allowed_extensions: List[str]) -> List[str]:
    paths = []
    for root, _, files in os.walk(folder):
        for file in files:
            ext = os.path.splitext(file)[1].lower()
            if ext in allowed_extensions or ext == "":
                paths.append(os.path.join(root, file))
    return paths

def read_file_safely(path: str) -> str:
    encodings = ["utf-8", "latin-1", "cp1252", "utf-16"]
    for enc in encodings:
        try:
            with open(path, "r", encoding=enc, errors="ignore") as f:
                return f.read()
        except Exception:
            continue
    return ""

def load_sms_collection_file(file_path):
    texts = []
    labels = []
    with open(file_path, "r", encoding="latin-1") as f:
        for line in f:
            parts = line.strip().split("\t", 1)
            if len(parts) != 2:
                continue
            label_text, message = parts
            message = clean_text(message)
            if label_text.lower() == "spam":
                label = 0
            elif label_text.lower() == "ham":
                label = 1
            else:
                continue
            texts.append(message)
            labels.append(label)
    df = pd.DataFrame({"text": texts, "label": labels})
    df["label_name"] = df["label"].map({0: "spam", 1: "ham"})
    print(f"\n[sms] Total samples: {len(df)}")
    print(f"[sms] spam (label=0): {(df['label'] == 0).sum()}")
    print(f"[sms] ham  (label=1): {(df['label'] == 1).sum()}")
    return df

def load_folder_dataset(dataset_root: str, dataset_name: str) -> pd.DataFrame:
    spam_dir = os.path.join(dataset_root, "spam")
    ham_dir = os.path.join(dataset_root, "ham")
    assert os.path.isdir(spam_dir), f"Missing folder: {spam_dir}"
    assert os.path.isdir(ham_dir), f"Missing folder: {ham_dir}"

    spam_files = list_text_files(spam_dir, CONFIG["allowed_extensions"])
    ham_files = list_text_files(ham_dir, CONFIG["allowed_extensions"])
    records = []

    for path in tqdm(spam_files, desc=f"Loading {dataset_name} spam"):
        txt = clean_text(read_file_safely(path))
        if txt:
            records.append({"text": txt, "label": 0, "label_name": "spam", "path": path})

    for path in tqdm(ham_files, desc=f"Loading {dataset_name} ham"):
        txt = clean_text(read_file_safely(path))
        if txt:
            records.append({"text": txt, "label": 1, "label_name": "ham", "path": path})

    df = pd.DataFrame(records).drop_duplicates(subset=["text"]).reset_index(drop=True)
    print(f"\n[{dataset_name}] Total samples: {len(df)}")
    print(f"[{dataset_name}] spam (label=0): {(df['label'] == 0).sum()}")
    print(f"[{dataset_name}] ham  (label=1): {(df['label'] == 1).sum()}")
    return df

def stratified_split(df: pd.DataFrame, seed: int = 42):
    train_df, temp_df = train_test_split(df, test_size=(1 - CONFIG["train_size"]), stratify=df["label"], random_state=seed)
    val_ratio_from_temp = CONFIG["val_size"] / (CONFIG["val_size"] + CONFIG["test_size"])
    val_df, test_df = train_test_split(temp_df, test_size=(1 - val_ratio_from_temp), stratify=temp_df["label"], random_state=seed)
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


## Cell 7 — Load and split datasets

In [ ]:

sms_df = load_sms_collection_file(CONFIG["sms_file"])
enron_df = load_folder_dataset(CONFIG["enron_root"], "enron")

sms_train, sms_val, sms_test = stratified_split(sms_df, CONFIG["seed"])
enron_train, enron_val, enron_test = stratified_split(enron_df, CONFIG["seed"])

print("SMS split sizes:", len(sms_train), len(sms_val), len(sms_test))
print("Enron split sizes:", len(enron_train), len(enron_val), len(enron_test))


## Cell 8 — Tokenizer and dataloaders

In [ ]:

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

class TextDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def create_dataloaders(train_df, val_df, test_df, tokenizer, max_len, batch_size):
    train_ds = TextDataset(train_df, tokenizer, max_len)
    val_ds = TextDataset(val_df, tokenizer, max_len)
    test_ds = TextDataset(test_df, tokenizer, max_len)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)
    return train_loader, val_loader, test_loader


## Cell 9 — Local Attention model

In [ ]:

class LocalBertSelfAttention(BertSelfAttention):
    def __init__(self, config, position_embedding_type=None, window_size=5):
        super().__init__(config, position_embedding_type=position_embedding_type)
        self.window_size = window_size

    def _distance_mask(self, seq_len, device, dtype):
        idx = torch.arange(seq_len, device=device)
        dist = (idx[None, :] - idx[:, None]).abs()
        allowed = dist <= self.window_size
        mask = torch.zeros(seq_len, seq_len, device=device, dtype=dtype)
        mask = mask.masked_fill(~allowed, torch.finfo(dtype).min)
        return mask.view(1, 1, seq_len, seq_len)

    def forward(self, hidden_states, attention_mask=None, head_mask=None, encoder_hidden_states=None, encoder_attention_mask=None, past_key_value=None, output_attentions=False):
        mixed_query_layer = self.query(hidden_states)
        is_cross_attention = encoder_hidden_states is not None
        current_states = encoder_hidden_states if is_cross_attention else hidden_states
        applied_attention_mask = encoder_attention_mask if is_cross_attention else attention_mask

        key_layer = self.transpose_for_scores(self.key(current_states))
        value_layer = self.transpose_for_scores(self.value(current_states))
        query_layer = self.transpose_for_scores(mixed_query_layer)

        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)

        seq_len = attention_scores.size(-1)
        attention_scores = attention_scores + self._distance_mask(seq_len, attention_scores.device, attention_scores.dtype)

        if applied_attention_mask is not None:
            attention_scores = attention_scores + applied_attention_mask

        attention_probs = nn.functional.softmax(attention_scores, dim=-1)
        attention_probs = self.dropout(attention_probs)

        if head_mask is not None:
            attention_probs = attention_probs * head_mask

        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        context_layer = context_layer.view(context_layer.size()[:-2] + (self.all_head_size,))

        outputs = (context_layer, attention_probs) if output_attentions else (context_layer,)
        if self.is_decoder:
            outputs = outputs + ((key_layer, value_layer),)
        return outputs

class InternalLocalAttentionBERT(BertPreTrainedModel):
    def __init__(self, config, window_size=5):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.bert = BertModel(config)
        classifier_dropout = config.classifier_dropout if getattr(config, "classifier_dropout", None) is not None else config.hidden_dropout_prob
        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        for i in range(len(self.bert.encoder.layer)):
            self.bert.encoder.layer[i].attention.self = LocalBertSelfAttention(
                config,
                position_embedding_type=getattr(config, "position_embedding_type", None),
                window_size=window_size,
            )
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output if outputs.pooler_output is not None else outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(pooled))
        loss = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return {"loss": loss, "logits": logits}

def base_raw_config(num_labels):
    return BertConfig(
        vocab_size=CONFIG["vocab_size"],
        hidden_size=CONFIG["hidden_size"],
        num_hidden_layers=CONFIG["num_hidden_layers"],
        num_attention_heads=CONFIG["num_attention_heads"],
        intermediate_size=CONFIG["intermediate_size"],
        hidden_dropout_prob=CONFIG["hidden_dropout_prob"],
        attention_probs_dropout_prob=CONFIG["attention_probs_dropout_prob"],
        num_labels=num_labels,
    )

def build_raw_local_model(num_labels, window_size=5):
    return InternalLocalAttentionBERT(base_raw_config(num_labels), window_size=window_size)


## Cell 10 — Train and compute F1

In [ ]:

def get_optimizer_and_scheduler(model, train_loader_len):
    optimizer = AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    total_steps = train_loader_len * CONFIG["epochs"]
    warmup_steps = int(total_steps * CONFIG["warmup_ratio"])
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    return optimizer, scheduler

def run_epoch(model, loader, optimizer, scheduler, device, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for batch in tqdm(loader, leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        if train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs["loss"]
            logits = outputs["logits"]

            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), CONFIG["max_grad_norm"])
                optimizer.step()
                scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_labels.extend(labels.detach().cpu().numpy().tolist())

    return {"loss": total_loss / len(loader), "f1": f1_score(all_labels, all_preds, pos_label=1)}

def train_and_test_local_window(dataset_name, train_df, val_df, test_df, max_len, window_size):
    train_loader, val_loader, test_loader = create_dataloaders(train_df, val_df, test_df, tokenizer, max_len, CONFIG["batch_size"])
    model = build_raw_local_model(CONFIG["num_labels"], window_size=window_size).to(CONFIG["device"])
    optimizer, scheduler = get_optimizer_and_scheduler(model, len(train_loader))

    best_val_f1 = -1.0
    best_state = None

    for epoch in range(CONFIG["epochs"]):
        print(f"{dataset_name} | window={window_size} | epoch {epoch+1}/{CONFIG['epochs']}")
        train_metrics = run_epoch(model, train_loader, optimizer, scheduler, CONFIG["device"], train=True)
        val_metrics = run_epoch(model, val_loader, optimizer, scheduler, CONFIG["device"], train=False)
        print("train f1:", round(train_metrics["f1"], 4), "| val f1:", round(val_metrics["f1"], 4))

        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    test_metrics = run_epoch(model, test_loader, optimizer=None, scheduler=None, device=CONFIG["device"], train=False)

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "dataset": dataset_name,
        "model": "local_attention",
        "window_size": window_size,
        "test_f1": test_metrics["f1"],
    }


## Cell 11 — Run ablation

In [ ]:

results = []

for w in CONFIG["window_sizes"]:
    results.append(train_and_test_local_window("sms", sms_train, sms_val, sms_test, CONFIG["sms_max_len"], w))

for w in CONFIG["window_sizes"]:
    results.append(train_and_test_local_window("enron", enron_train, enron_val, enron_test, CONFIG["enron_max_len"], w))

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(CONFIG["output_root"], "local_attention_window_ablation_f1.csv"), index=False)

print(results_df)


## Cell 12 — Pivot view

In [ ]:

pivot_df = results_df.pivot(index="window_size", columns="dataset", values="test_f1")
display(pivot_df)
